# Automatización Meme Reaction — Documentación

**Pipeline automatizado end-to-end para generar videos de meme reaction y subirlos a redes.**

Objetivo: Scraping de memes → Descarga → Clasificación IA → Match con clip de reacción → Verificación IA → Caption IA → Generación de video → (futuro: upload)

Todo este proyecto vive en: `automatizaciones/Meme_Reaction/`

## Pipeline General (Pasos)

```
1. SCRAPING (Selenium)
   Abre navegador → entra a IG → navega perfiles target → copia links de posts tipo foto

2. DESCARGA (instaloader, SIN LOGIN)
   Descarga las fotos usando los links obtenidos en paso 1

3. CLASIFICACIÓN IA (OpenAI Vision)
   Analiza cada meme y lo categoriza (tipo de humor/remate)

4. MATCH CON CLIP DE REACCIÓN
   Según la categoría del meme, elige un clip de reacción pre-catalogado

5. VERIFICACIÓN IA
   Verifica que la descripción del clip haga sentido con el meme

6. CAPTION IA
   Decide si vale la pena un caption y lo genera

7. GENERACIÓN DE VIDEO
   Meme arriba + Clip abajo + Caption (si aplica)
   Audio: SIEMPRE el del clip (no se puede cambiar en automatización)

8. GUARDAR JSON CONFIG
   Se guarda la combinación para replicar/editar manualmente después

9. (FUTURO) UPLOAD
   Subir a redes automáticamente
```

## `main.py` — Orquestador

**Qué hace:**
1. `pip install -r requirements.txt` (instala deps del sub-proyecto)
2. Carga `.env` del proyecto general (`../../.env`) con python-dotenv
3. Ejecuta los scripts numerados en orden: `1_` → `2_` → ... → `8_`
4. Si un script no existe (placeholder), lo salta con aviso
5. Al final muestra resumen de qué pasos pasaron y cuáles no

**Uso:**
```bash
python automatizaciones/Meme_Reaction/main.py
```

**Nota:** El `main.py` ejecuta cada paso como subprocess independiente. Cada `N_*.py` debe funcionar de forma standalone también (para debug individual).

---

## `requirements.txt` — Dependencias del Sub-Proyecto

Solo incluye lo que ESTE pipeline necesita (no todo el proyecto general):

| Grupo | Paquetes |
| --- | --- |
| Scraping | selenium, webdriver-manager |
| Instagram | instaloader |
| IA | openai, python-dotenv |
| Video | moviepy, Pillow, numpy |
| General | requests |

**Nota:** NO incluye yt-dlp, instagrapi, ni google-api (esos son del proyecto general, no de esta automatización).

## Paso 1: Scraping de Links con Selenium

**Archivo:** `1_scrape_meme_links.py` ✅ IMPLEMENTADO

**Navegador:** Brave (Chromium-based, usa ChromeDriver via webdriver-manager)

**Flujo:**
1. Abre Brave VISIBLE (no headless) → navega a instagram.com
2. **PAUSA PARA LOGIN MANUAL** → espera Enter del usuario
3. Navega a cada perfil target
4. Scrollea N veces para cargar posts (configurable)
5. Extrae todos los `<a href="/p/XXXXX/">` del grid
6. Filtra: solo fotos (skip videos/reels que tienen SVG overlay)
7. Filtra: skip shortcodes ya en historial
8. Guarda nuevos en `historial/links_scrapeados.json`

**Login manual:**
- Selenium NUNCA hace login programático
- Abre el browser, pausa en terminal con `input()`
- Tú haces login manualmente si Instagram lo pide
- Presionas Enter para continuar

**Cómo distingue fotos de videos:**
- En el grid de IG, los videos/reels tienen un SVG overlay con `aria-label` que contiene "Reel" o "Video"
- Las fotos puras no tienen ningún SVG overlay
- Se filtra por ausencia de esos SVGs

**Anti-detección:**
- `--disable-blink-features=AutomationControlled`
- Quita `navigator.webdriver`
- Delays aleatorios entre scrolls
- Ventana visible (no headless)

**Configuración (en el archivo):**
- `BRAVE_PATH` — ruta al ejecutable de Brave
- `PERFILES_TARGET` — lista de usernames a scrapear
- `SCROLL_COUNT` — cuántas veces scrollear (default: 5)
- `SCROLL_DELAY` — segundos entre scrolls (default: 2.5)

**Output:** `historial/links_scrapeados.json`
```json
{
  "scrapeados": ["shortcode1", "shortcode2", ...],
  "por_descargar": ["shortcode1", "shortcode2", ...]
}
```

**Uso:**
```bash
python 1_scrape_meme_links.py
```

## Paso 1: Scraping de Links con Selenium

**Archivo:** `scrape_meme_links.py` (por crear)

**Funcionalidad:**
- Recibe lista de perfiles target (configurable)
- Abre Chrome (headless o visible, configurable)
- Para cada perfil:
  - Navega a `instagram.com/{username}/`
  - Scrollea para cargar posts
  - Identifica posts tipo foto (vs video/reel)
  - Extrae shortcodes de los links (`/p/XXXXX/`)
  - Filtra posts ya descargados (historial)
- Output: lista de shortcodes nuevos

**Consideraciones técnicas (por investigar):**
- Selenium puede necesitar webdriver (chromedriver)
- Instagram puede cargar posts con lazy loading (infinite scroll)
- Distinguir fotos de videos en el grid puede requerir inspeccionar el DOM
- Rate limiting: cuántos perfiles/scrolls por sesión
- Headless vs visible: headless es más rápido pero puede ser más detectable

**Dependencias:**
```
selenium
webdriver-manager  # Para auto-instalar chromedriver
```

**Estado:** POR IMPLEMENTAR

## Paso 2: Descarga de Posts

**Archivo:** `download_memes.py` (por crear)

**Funcionalidad:**
- Recibe lista de shortcodes del Paso 1
- Usa `instaloader` SIN LOGIN
- Descarga cada foto con delay entre descargas
- Guarda en carpeta de memes descargados
- Actualiza historial (no repetir descargas)

**Carpeta de descarga:** `automatizaciones/Meme_Reaction/memes_descargados/` (por definir)

**Historial:** JSON con shortcodes ya procesados

**Estado:** POR IMPLEMENTAR (reutilizar lógica de `tools/instagram/single_nologin.py`)

## Paso 3: Clasificación del Meme con IA

**Archivo:** `classify_meme.py` (por crear)

**Funcionalidad:**
- Recibe path de imagen del meme
- Envía a OpenAI Vision API
- Prompt pide categorizar el "remate" del meme

**Categorías (propuesta inicial, por refinar):**

| Categoría | Descripción | Ejemplo de reacción |
| --- | --- | --- |
| `humor_absurdo` | Algo random/sin sentido que da risa | Clip de risa descontrolada |
| `humor_dark` | Humor negro/ofensivo | Clip de "ohhh" |
| `cringe` | Vergüenza ajena | Clip de taparse la cara |
| `sad_funny` | Triste pero de risa ("me identifiqué") | Clip de llorar de risa |
| `wholesome` | Tierno/bonito | Clip de sonrisa |
| `plot_twist` | Giro inesperado | Clip de sorpresa |
| `relatable` | Muy identificable ("yo") | Clip de asentir |
| `rage` | Enojo/frustración | Clip de golpear mesa |
| `sus` | Doble sentido / sexual | Clip de mirada cómplice |
| `intellectual` | Humor inteligente/referencial | Clip pensativo |

**Output:** Categoría + confianza + breve descripción de por qué

**Estado:** POR IMPLEMENTAR

## Paso 4: Match con Clip de Reacción

**Funcionalidad:**
- Tengo un catálogo de clips de reacción pre-categorizados
- Cada clip tiene: path, categoría(s), descripción manual de qué pasa en el video
- Según la categoría del meme (Paso 3), se elige un clip compatible
- Si hay varios clips para la misma categoría, se elige random o round-robin

**Catálogo de clips:** JSON en `automatizaciones/Meme_Reaction/catalogo_clips.json`

Estructura propuesta:
```json
{
  "clips": [
    {
      "id": "risa_01",
      "path": "tools_output/videos/risa_descontrolada (audio).mp4",
      "categorias": ["humor_absurdo", "sad_funny"],
      "descripcion": "Persona riéndose sin control hasta llorar",
      "usado_count": 0
    }
  ]
}
```

**Importante:** La descripción del clip la hago YO manualmente al catalogar. La IA NO analiza el video.

**Estado:** POR IMPLEMENTAR (primero necesito catalogar mis clips)

## Paso 5: Verificación IA (Meme + Clip)

**Funcionalidad:**
- Envía a la IA:
  - La imagen del meme
  - La descripción manual del clip elegido
  - La categoría asignada
- Pregunta: "¿Tiene sentido esta combinación? ¿El clip es buena reacción para este meme?"
- Si la IA dice que NO → se prueba otro clip de la misma categoría o se marca para revisión manual

**Output:** `aprobado` / `rechazado` + razón

**Estado:** POR IMPLEMENTAR

## Paso 6: Generación de Caption con IA

**Funcionalidad:**
- Envía a la IA:
  - La imagen del meme
  - La categoría
  - El clip elegido (descripción)
- Pregunta: "¿Este video necesita un caption superpuesto? Si sí, ¿cuál?"
- La IA puede responder:
  - `no_caption` — el meme habla por sí solo
  - `caption: "texto aquí"` — agregar este texto

**Reglas para el caption:**
- Corto (máximo 2 líneas)
- Usa `|` para salto de línea (patrón del generator)
- Puede ser meme text, reacción, o contexto

**Output:** caption string o None

**Estado:** POR IMPLEMENTAR

## Paso 7: Generación del Video

**Funcionalidad:**
- Usa la lógica de `generators/meme_reaction.py`
- Inputs: imagen del meme + clip de reacción + caption (opcional)
- **Audio: SIEMPRE el del clip** (no se puede elegir audio externo en automatización)
- Output: video final en `output/meme_reaction/`

**Diferencias con el generator manual:**
- No hay selección interactiva (todo viene del pipeline)
- Audio siempre del clip (no pregunta)
- Se guarda JSON config automáticamente (Paso 8)

**Estado:** POR IMPLEMENTAR (reutilizar `generate_meme_reaction()` de `generators/meme_reaction.py`)

## Paso 8: Guardar JSON Config

**Funcionalidad:**
- Después de generar el video, guarda un JSON con toda la combinación:

```json
{
  "meme_path": "automatizaciones/Meme_Reaction/memes_descargados/shortcode.jpg",
  "clip_path": "tools_output/videos/risa_01 (audio).mp4",
  "clip_id": "risa_01",
  "caption": "texto del caption|linea 2",
  "caption_size": "M",
  "categoria_meme": "humor_absurdo",
  "audio_source": "clip",
  "output_name": "meme_reaction_20260520_001.mp4",
  "generated_at": "2026-05-20T15:30:00",
  "ai_verification": "aprobado",
  "auto_generated": true
}
```

**Carpeta:** `automatizaciones/Meme_Reaction/configs_generados/`

**Para qué sirve:**
- Replicar el video manualmente si quiero cambiar algo
- Historial de qué se generó
- Poder cambiar el audio al re-generar manualmente

## Script Manual: Replicar/Editar desde JSON

**Archivo:** `manual_from_config.py` (por crear)

**Funcionalidad:**
- Carga un JSON config generado por la automatización
- Muestra qué tiene: meme, clip, caption, audio
- **Pregunta si quieres usar ese audio o elegir otro** (la diferencia con la automatización)
- Permite editar caption antes de generar
- Genera el video con los cambios

**Flujo:**
1. Lista JSONs disponibles en `configs_generados/`
2. Eliges uno
3. Te muestra preview de la combinación
4. "¿Usar audio del clip o elegir otro?" → si otro, usa browse_folder en audios/
5. "¿Editar caption?" → puedes cambiarlo o quitarlo
6. Genera video

**Estado:** POR IMPLEMENTAR

## Estructura de Carpetas

```
automatizaciones/Meme_Reaction/
├── DOCUMENTACION                # Este notebook
├── main.py                      # Orquestador: pip install + ejecuta pasos en orden
├── requirements.txt             # Dependencias SOLO de este sub-proyecto
├── 1_scrape_meme_links.py       # Paso 1: Selenium scraping de links
├── 2_download_memes.py          # Paso 2: Descarga con instaloader (sin login)
├── 3_classify_meme.py           # Paso 3: Clasificación IA del meme
├── 4_match_clip.py              # Paso 4: Match meme → clip de reacción
├── 5_verify_match.py            # Paso 5: Verificación IA (meme+clip)
├── 6_generate_caption.py        # Paso 6: Generación de caption con IA
├── 7_generate_video.py          # Paso 7: Generar video final
├── 8_save_config.py             # Paso 8: Guardar JSON config
├── manual_from_config.py        # Script manual (NO parte del pipeline)
├── catalogo_clips.json          # Catálogo de clips con categorías (por crear)
├── config.json                  # Config general (por crear)
├── memes_descargados/           # Fotos descargadas (Paso 2)
│   └── .gitkeep
├── configs_generados/           # JSONs de cada video generado (Paso 8)
│   └── .gitkeep
└── historial/                   # Control de no-repetición
    └── .gitkeep
```

**Nota:** `.env` NO se duplica aquí. Se usa el del proyecto general (`../../.env`).

## Config General (`config.json`)

Configuraciones del proyecto (propuesta):

```json
{
  "perfiles_target": [
    "elmello2023",
    "otro_perfil_memes"
  ],
  "max_posts_por_perfil": 10,
  "delay_entre_descargas": 5,
  "delay_entre_perfiles": 30,
  "selenium": {
    "headless": true,
    "scroll_count": 5,
    "scroll_delay": 2
  },
  "openai": {
    "model": "gpt-4o",
    "max_tokens": 500
  },
  "output_dir": "output/meme_reaction/",
  "caption_default_size": "M"
}
```

## Reglas del Proyecto

1. **NUNCA login en Instagram** — solo Selenium (scraping) + instaloader (descarga sin login)
2. **Audio en automatización = audio del clip SIEMPRE** — no se cambia
3. **Audio en manual (desde JSON) = se pregunta** — puede ser del clip o externo
4. **Cada cambio en código debe reflejarse en esta documentación**
5. **JSON config se guarda SIEMPRE** después de generar un video
6. **Clips se catalogan manualmente** — la IA no analiza videos, solo usa la descripción que yo escribo
7. **Historial de descargas** — nunca repetir un post ya descargado
8. **Verificación IA** — si no pasa, se busca otro clip o se marca para revisión manual

## Estado Actual

| Componente | Estado |
| --- | --- |
| Documentación | ✅ Creada |
| Estructura de carpetas | ✅ Creada (.gitkeep en cada dir) |
| `main.py` (orquestador) | ✅ Creado (skeleton funcional) |
| `requirements.txt` | ✅ Creado |
| `1_scrape_meme_links.py` | ✅ Implementado (Brave + login manual + filtro fotos) |
| `2_download_memes.py` | ⏳ Placeholder (por implementar) |
| `3_classify_meme.py` | ⏳ Placeholder (por implementar) |
| `4_match_clip.py` | ⏳ Placeholder (por implementar) |
| `5_verify_match.py` | ⏳ Placeholder (por implementar) |
| `6_generate_caption.py` | ⏳ Placeholder (por implementar) |
| `7_generate_video.py` | ⏳ Placeholder (por implementar) |
| `8_save_config.py` | ⏳ Placeholder (por implementar) |
| `manual_from_config.py` | ⏳ Placeholder (por implementar) |
| `catalogo_clips.json` | ⏳ Por crear (necesito catalogar clips) |
| `config.json` | ⏳ Por crear |

**Próximos pasos:**
1. ✅ ~~Investigar Selenium para scraping de Instagram~~
2. ✅ ~~Implementar `1_scrape_meme_links.py`~~
3. Probar `1_scrape_meme_links.py` en local
4. Implementar `2_download_memes.py`
5. Definir y refinar categorías de memes
6. Empezar a catalogar clips de reacción
7. Implementar pasos 3-8
8. Integrar todo via `main.py`

## Setup Local (Primera Vez)

### 1. Prerrequisitos
- Python 3.10+ instalado
- **Brave Browser** instalado (ruta default: `C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe`)
- Git (si clonas el repo)

### 2. Clonar/Sincronizar el proyecto
Si ya tienes el repo en local, solo asegúrate de que la carpeta `automatizaciones/Meme_Reaction/` exista con todos los archivos.

### 3. Crear entorno virtual (recomendado)
```bash
cd drako-edits/automatizaciones/Meme_Reaction
python -m venv venv

# Windows:
venv\Scripts\activate

# Mac/Linux:
source venv/bin/activate
```

### 4. Instalar dependencias
```bash
pip install -r requirements.txt
```

### 5. Configurar `.env`
El `.env` vive en la raíz del proyecto general (`drako-edits/.env`). Debe tener:
```env
# OpenAI
OPENAI_API_KEY=sk-...

# Instagram (ya NO se usa login programático)
IG_USERNAME=
IG_PASSWORD=
```

### 6. Verificar Brave + Selenium
```bash
python -c "from selenium import webdriver; from webdriver_manager.chrome import ChromeDriverManager; print('OK')"
```
`webdriver-manager` descarga chromedriver automáticamente según tu versión de Brave.

### 7. Primer test
```bash
# Probar que el main corre (saltará todos los pasos porque son placeholder)
python main.py
```

### 8. Probar Selenium (Paso 1)
```bash
python 1_scrape_meme_links.py
```
Debería abrir Brave, navegar a IG, pausar para login, y luego scrapear.

---

### Orden de implementación para ir probando:
```
1. python 1_scrape_meme_links.py   ← probar Selenium con 1 perfil
2. python 2_download_memes.py      ← probar descarga de 1 foto
3. python 3_classify_meme.py       ← probar con 1 imagen local
4. python 4_match_clip.py          ← necesitas catalogo_clips.json primero
5. python 5_verify_match.py        ← probar con 1 combo
6. python 6_generate_caption.py    ← probar con 1 combo
7. python 7_generate_video.py      ← probar generación completa
8. python 8_save_config.py         ← verificar que grabe JSON
9. python main.py                  ← correr todo junto
```

Cada script debe funcionar **standalone** (lo corres individual para debug) Y también ser llamado por `main.py` en secuencia.